# 🦆 Duck Egg Fertility Detection — Training Notebook (Google Colab)
**Pipeline:** Preprocessing → U-Net Segmentation → Feature Extraction → AWC + Baselines → Evaluation

---
### Cara pakai:
1. Upload folder `data/` ke Google Drive kamu
2. Jalankan cell **satu per satu dari atas ke bawah** (Runtime → Run all juga bisa)
3. Semua hasil otomatis tersimpan di Drive, aman walau sesi Colab putus

### Struktur Drive yang dibutuhkan:
```
MyDrive/duck_egg_fertility/
└── data/
    ├── fertile/        ← gambar telur fertil (*.jpg)
    └── infertile/      ← gambar telur infertil (*.jpg)
```


## ⚙ Setup Awal
### Cell 1 — Mount Google Drive

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT  = '/content/drive/MyDrive/duck_egg_fertility'
REPO_DIR    = '/content/duck_egg_fertility_detection'
DATA_RAW    = os.path.join(DRIVE_ROOT, 'data')

# buat folder drive jika belum ada
os.makedirs(os.path.join(DRIVE_ROOT, 'models'),  exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, 'results'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_ROOT, 'data'),    exist_ok=True)

print('Drive mounted.')
print('DRIVE_ROOT :', DRIVE_ROOT)
print('Folder data:', DATA_RAW)


### Cell 2 — Cek GPU

In [ ]:
# ── Cek GPU ──────────────────────────────────────────────────────────────────
import torch
print('GPU tersedia:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2), 'GB')
else:
    print('⚠ Tidak ada GPU — aktifkan di Runtime → Change runtime type → T4 GPU')


### Cell 3 — Clone / Setup Repo

In [ ]:
# ── Clone repo & install dependencies ────────────────────────────────────────
import os, subprocess

REPO_DIR = '/content/duck_egg_fertility_detection'

if not os.path.exists(REPO_DIR):
    # Opsi A: clone dari GitHub (ganti URL dengan repo kamu)
    # !git clone https://github.com/USERNAME/duck_egg_fertility_detection.git $REPO_DIR

    # Opsi B: copy dari Drive (jika repo sudah di-zip ke Drive)
    drive_zip = '/content/drive/MyDrive/duck_egg_fertility/repo.zip'
    if os.path.exists(drive_zip):
        os.makedirs(REPO_DIR, exist_ok=True)
        os.system(f'unzip -q {drive_zip} -d {REPO_DIR}')
        print('Repo di-unzip dari Drive.')
    else:
        print('⚠ Repo belum ada. Pilih salah satu:')
        print('  1. Uncomment baris git clone di atas, atau')
        print('  2. Upload repo.zip ke Drive di path:', drive_zip)
else:
    print('Repo sudah ada:', REPO_DIR)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())


### Cell 4 — Install Dependencies

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
# Colab sudah punya torch, numpy, sklearn — install yang kurang saja
import subprocess, sys

packages = [
    'opencv-python-headless',
    'scikit-image',
    'imageio',
    'PyYAML',
    'loguru',
    'scikit-fuzzy',
    'tqdm',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
    print(f'  ✓ {pkg}')

print('\nSemua dependencies terinstall.')


---
## 📁 Persiapan Data
### Cell 5 — Setup & Distribusi Data

In [ ]:
# ── Setup struktur data ───────────────────────────────────────────────────────
import os, shutil, glob

REPO_DIR   = '/content/duck_egg_fertility_detection'
DRIVE_ROOT = '/content/drive/MyDrive/duck_egg_fertility'
DATA_RAW   = os.path.join(REPO_DIR, 'data', 'raw')

# Buat folder struktur yang dibutuhkan
for split in ['train', 'val', 'test']:
    for cls in ['fertile', 'infertile']:
        os.makedirs(os.path.join(DATA_RAW, split, cls), exist_ok=True)

# ── Cek apakah data sudah ada di repo ────────────────────────────────────────
n_fertile   = len(glob.glob(os.path.join(DATA_RAW, '**', 'fertile',   '*.jpg'), recursive=True))
n_infertile = len(glob.glob(os.path.join(DATA_RAW, '**', 'infertile', '*.jpg'), recursive=True))
print(f'Data di repo: {n_fertile} fertile, {n_infertile} infertile')

# ── Jika data masih di Drive (flat folder), salin ke repo ────────────────────
drive_fertile   = os.path.join(DRIVE_ROOT, 'data', 'fertile')
drive_infertile = os.path.join(DRIVE_ROOT, 'data', 'infertile')

if n_fertile == 0 and os.path.exists(drive_fertile):
    imgs = sorted(glob.glob(os.path.join(drive_fertile,   '*.jpg')))
    n    = len(imgs)
    n_tr, n_va = int(n*0.70), int(n*0.15)
    splits = [('train',imgs[:n_tr]), ('val',imgs[n_tr:n_tr+n_va]), ('test',imgs[n_tr+n_va:])]
    for split, files in splits:
        for f in files:
            shutil.copy2(f, os.path.join(DATA_RAW, split, 'fertile'))
    print(f'Fertile disalin: {n} gambar → train/val/test')

if n_infertile == 0 and os.path.exists(drive_infertile):
    imgs = sorted(glob.glob(os.path.join(drive_infertile, '*.jpg')))
    n    = len(imgs)
    n_tr, n_va = int(n*0.70), int(n*0.15)
    splits = [('train',imgs[:n_tr]), ('val',imgs[n_tr:n_tr+n_va]), ('test',imgs[n_tr+n_va:])]
    for split, files in splits:
        for f in files:
            shutil.copy2(f, os.path.join(DATA_RAW, split, 'infertile'))
    print(f'Infertile disalin: {n} gambar → train/val/test')

# ── Ringkasan ─────────────────────────────────────────────────────────────────
print('\n── Distribusi Data ──────────────────────────')
for split in ['train', 'val', 'test']:
    nf = len(glob.glob(os.path.join(DATA_RAW, split, 'fertile',   '*.jpg')))
    ni = len(glob.glob(os.path.join(DATA_RAW, split, 'infertile', '*.jpg')))
    print(f'  {split:5s}: {nf:3d} fertile  {ni:3d} infertile  total={nf+ni}')


---
## 🔬 Step 1 — Preprocessing

In [ ]:
# ── Step 1: Preprocessing ─────────────────────────────────────────────────────
# CLAHE + Homomorphic Filtering + Bilateral Denoising → 256×256
import subprocess, sys, os

REPO_DIR = '/content/duck_egg_fertility_detection'
os.chdir(REPO_DIR)

result = subprocess.run(
    [sys.executable, 'scripts/01_preprocess_data.py',
     '--input-dir',  'data/raw',
     '--output-dir', 'data/preprocessed',
     '--preset',     'default'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Preprocessing gagal — cek error di atas')
print('\n✓ Preprocessing selesai → data/preprocessed/')


---
## 🎭 Step 2 — Generate Segmentation Masks

In [ ]:
# ── Step 2: Generate Segmentation Masks ──────────────────────────────────────
# Buat mask untuk training U-Net (thresholding Otsu sebagai pseudo-mask awal)
import subprocess, sys, os

REPO_DIR = '/content/duck_egg_fertility_detection'
os.chdir(REPO_DIR)

result = subprocess.run(
    [sys.executable, 'scripts/03b_generate_masks.py',
     '--data-dir', 'data/preprocessed',
     '--out-dir',  'data/segmentation'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Mask generation gagal')
print('\n✓ Masks dibuat → data/segmentation/')


---
## 🧠 Step 3 — Training U-Net Segmentation
### Cell 8 — Konfigurasi (edit di sini sebelum training)

In [ ]:
# ── Step 3a: Konfigurasi U-Net ───────────────────────────────────────────────
# Edit parameter di sini sebelum training

UNET_CONFIG = {
    'model': {
        'n_channels':    3,
        'n_classes':     3,
        'bilinear':      True,
        'dropout_rate':  0.3,
        'lightweight':   True,
    },
    'training': {
        'batch_size':              8,
        'num_epochs':              100,     # max epoch (early stop patience=15)
        'learning_rate':           0.001,
        'weight_decay':            0.0001,
        'loss_type':               'ce',
        'seed':                    42,
        'early_stopping_patience': 15,
        'checkpoint_interval':     10,
        'lr_patience':             5,
        'lr_reduce_factor':        0.5,
    },
    'data': {
        'image_size':   [256, 256],
        'num_workers':  2,
        'pin_memory':   True,
        'train_images': 'data/segmentation/train/images',
        'train_masks':  'data/segmentation/train/masks',
        'val_images':   'data/segmentation/val/images',
        'val_masks':    'data/segmentation/val/masks',
    },
}

import yaml, os
os.chdir('/content/duck_egg_fertility_detection')
os.makedirs('configs', exist_ok=True)
with open('configs/unet_config_colab.yaml', 'w') as f:
    yaml.dump(UNET_CONFIG, f, default_flow_style=False)

print('Config U-Net tersimpan → configs/unet_config_colab.yaml')
print()
for section, params in UNET_CONFIG.items():
    print(f'[{section}]')
    for k, v in params.items():
        print(f'  {k}: {v}')
    print()


### Cell 9 — Jalankan Training U-Net

In [ ]:
# ── Step 3b: Train U-Net ─────────────────────────────────────────────────────
# ⏱ Estimasi waktu: ~15-30 menit di T4 GPU (100 epoch, early stop ~75 epoch)
import subprocess, sys, os, time

REPO_DIR   = '/content/duck_egg_fertility_detection'
DRIVE_ROOT = '/content/drive/MyDrive/duck_egg_fertility'
os.chdir(REPO_DIR)

t0 = time.time()
result = subprocess.run(
    [sys.executable, 'scripts/03_train_unet.py',
     '--config', 'configs/unet_config_colab.yaml'],
    capture_output=True, text=True
)

elapsed = time.time() - t0
print(f'Training selesai dalam {elapsed/60:.1f} menit')
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('U-Net training gagal')

# ── Salin model ke Drive ──────────────────────────────────────────────────────
import shutil, glob
os.makedirs(os.path.join(DRIVE_ROOT, 'models', 'unet'), exist_ok=True)
for f in glob.glob('models/unet/*.pth') + glob.glob('models/unet/*.json'):
    dst = os.path.join(DRIVE_ROOT, 'models', 'unet', os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'  Disimpan ke Drive: {dst}')

print('\n✓ U-Net training selesai & model tersimpan di Drive')


---
## 🔍 Step 4 — Ekstraksi Fitur

In [ ]:
# ── Step 4: Ekstraksi Fitur (Classical + Deep Embedding) ─────────────────────
# ⏱ Estimasi: ~5-10 menit
import subprocess, sys, os, shutil, glob, time

REPO_DIR   = '/content/duck_egg_fertility_detection'
DRIVE_ROOT = '/content/drive/MyDrive/duck_egg_fertility'
os.chdir(REPO_DIR)

t0 = time.time()
result = subprocess.run(
    [sys.executable, 'scripts/04_extract_features.py',
     '--data-root',       'data',
     '--output-dir',      'data/features',
     '--mode',            'hybrid',
     '--unet-checkpoint', 'models/unet/unet_best.pth'],
    capture_output=True, text=True
)
elapsed = time.time() - t0
print(f'Feature extraction selesai dalam {elapsed/60:.1f} menit')
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('Feature extraction gagal')

# ── Salin features ke Drive ───────────────────────────────────────────────────
import numpy as np
os.makedirs(os.path.join(DRIVE_ROOT, 'data', 'features'), exist_ok=True)
for f in glob.glob('data/features/*.npy') + glob.glob('data/features/*.json'):
    dst = os.path.join(DRIVE_ROOT, 'data', 'features', os.path.basename(f))
    shutil.copy2(f, dst)

# ── Ringkasan fitur ───────────────────────────────────────────────────────────
try:
    X_tr = np.load('data/features/awc_features.npy')
    y_tr = np.load('data/features/awc_labels.npy')
    X_te = np.load('data/features/awc_test_features.npy')
    y_te = np.load('data/features/awc_test_labels.npy')
    print(f'\nTrain features : {X_tr.shape}  labels: {y_tr.shape}')
    print(f'Test  features : {X_te.shape}  labels: {y_te.shape}')
    print(f'Feature dim    : {X_tr.shape[1]}')
    print(f'Kelas train    : fertile={int((y_tr==0).sum())}  infertile={int((y_tr==1).sum())}')
except FileNotFoundError:
    print('Fitur belum ditemukan — cek error di atas')

print('\n✓ Features tersimpan di data/features/ & Drive')


---
## 🔵 Step 5 — Training AWC & Baselines
### Cell 11 — Konfigurasi AWC

In [ ]:
# ── Step 5a: Konfigurasi AWC ──────────────────────────────────────────────────
AWC_CONFIG = {
    'data': {
        'n_clusters':     2,
        'normalize':      True,
        'standardize':    True,
        'train_features': 'data/features/awc_features.npy',
        'train_labels':   'data/features/awc_labels.npy',
        'test_features':  'data/features/awc_test_features.npy',
        'test_labels':    'data/features/awc_test_labels.npy',
    },
    'algorithm': {
        'max_iter':               100,
        'tol':                    0.0001,
        'random_state':           42,
        'weighted_centroids':     True,
        'weight_update_interval': 5,
    },
    'advanced': {
        'feature_selection': True,
        'selection_method':  'anova',
        'k_best':            20,
        'variance_threshold':0.1,
    },
    'output': {
        'model_path':   'models/awc/awc_model.pkl',
        'metrics_path': 'results/awc_evaluation/metrics.json',
    },
}

import yaml, os
os.makedirs('configs', exist_ok=True)
with open('configs/awc_config_colab.yaml', 'w') as f:
    yaml.dump(AWC_CONFIG, f, default_flow_style=False)
print('Config AWC tersimpan → configs/awc_config_colab.yaml')


### Cell 12 — Jalankan Training AWC + K-Means + FCM

In [ ]:
# ── Step 5b: Train AWC + K-Means + FCM ───────────────────────────────────────
import subprocess, sys, os, shutil, glob, time

REPO_DIR   = '/content/duck_egg_fertility_detection'
DRIVE_ROOT = '/content/drive/MyDrive/duck_egg_fertility'
os.chdir(REPO_DIR)

# ── AWC ───────────────────────────────────────────────────────────────────────
print('Training AWC...')
t0 = time.time()
res = subprocess.run(
    [sys.executable, 'scripts/05_train_awc.py',
     '--config', 'configs/awc_config_colab.yaml'],
    capture_output=True, text=True
)
print(f'AWC selesai: {time.time()-t0:.1f}s')
print(res.stdout[-2000:] if len(res.stdout) > 2000 else res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr[-1000:])

# ── Salin semua model ke Drive ────────────────────────────────────────────────
for src_glob, drive_sub in [
    ('models/awc/*',       'models/awc'),
    ('models/baselines/*', 'models/baselines'),
]:
    os.makedirs(os.path.join(DRIVE_ROOT, drive_sub), exist_ok=True)
    for f in glob.glob(src_glob):
        if os.path.isfile(f):
            shutil.copy2(f, os.path.join(DRIVE_ROOT, drive_sub, os.path.basename(f)))

print('\n✓ Model AWC + baselines tersimpan di Drive')


---
## 📊 Step 6 — Evaluasi Model

In [ ]:
# ── Step 6: Evaluasi Semua Model ─────────────────────────────────────────────
import subprocess, sys, os, json, shutil, glob

REPO_DIR   = '/content/duck_egg_fertility_detection'
DRIVE_ROOT = '/content/drive/MyDrive/duck_egg_fertility'
os.chdir(REPO_DIR)

res = subprocess.run(
    [sys.executable, 'scripts/06_evaluate_models.py'],
    capture_output=True, text=True
)
print(res.stdout[-4000:] if len(res.stdout) > 4000 else res.stdout)
if res.returncode != 0:
    print('STDERR:', res.stderr[-1000:])

# ── Tampilkan ringkasan metrik ────────────────────────────────────────────────
metrics_path = 'results/awc_evaluation/metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        m = json.load(f)
    print('\n── Hasil Evaluasi ────────────────────────────────')
    for key, val in m.items():
        if isinstance(val, float):
            print(f'  {key:25s}: {val:.4f}')
        elif isinstance(val, dict):
            print(f'  [{key}]')
            for k2, v2 in val.items():
                if isinstance(v2, float):
                    print(f'    {k2:20s}: {v2:.4f}')

# ── Salin hasil ke Drive ──────────────────────────────────────────────────────
os.makedirs(os.path.join(DRIVE_ROOT, 'results'), exist_ok=True)
for f in glob.glob('results/**/*', recursive=True):
    if os.path.isfile(f):
        rel  = os.path.relpath(f, 'results')
        dst  = os.path.join(DRIVE_ROOT, 'results', rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(f, dst)

print('\n✓ Semua hasil evaluasi tersimpan di Drive/results/')


---
## 📈 Visualisasi Hasil Akhir

In [ ]:
# ── Visualisasi Hasil Akhir ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import json, os

REPO_DIR = '/content/duck_egg_fertility_detection'
os.chdir(REPO_DIR)

# ── Muat metrik dari berbagai sumber ──────────────────────────────────────────
def try_load(path):
    try:
        with open(path) as f: return json.load(f)
    except: return {}

awc_m = try_load('results/awc_evaluation/metrics.json')
eval_m = try_load('results/statistical_tests/eval_summary.json')

# ── Hasil dari paper sebagai referensi ───────────────────────────────────────
PAPER = {
    'AWC':     {'acc':0.891,'prec':0.884,'rec':0.903,'f1':0.893,'auc':0.937},
    'K-Means': {'acc':0.814,'prec':0.807,'rec':0.821,'f1':0.814,'auc':0.863},
    'FCM':     {'acc':0.836,'prec':0.831,'rec':0.843,'f1':0.837,'auc':0.902},
}

metrics    = ['acc','prec','rec','f1','auc']
met_labels = ['Accuracy','Precision','Recall','F1-Score','AUC']
colors     = ['#C0392B','#2980B9','#27AE60']
x          = np.arange(len(metrics)); w = 0.25

fig, axes = plt.subplots(2, 1, figsize=(8, 11))
fig.suptitle('Hasil Training - AWC vs Baselines (Nilai dari Paper sebagai referensi)',
             fontsize=13, fontweight='bold')

# Bar chart referensi paper
ax = axes[0]
for i, (method, mdata) in enumerate(PAPER.items()):
    vals = [mdata[m] for m in metrics]
    ax.bar(x + i*w - w, vals, w*0.88, label=method,
           color=colors[i], alpha=0.85, edgecolor='black', linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(met_labels, fontsize=10)
ax.set_ylabel('Score'); ax.set_ylim(0.70, 1.02)
ax.set_title('(a) Referensi Paper (5-Fold CV)', fontsize=11)
ax.legend(); ax.grid(alpha=0.2, axis='y', linestyle=':')

# Hasil aktual dari training ini (jika ada)
ax = axes[1]
trained_results = {}
# Coba baca dari eval_summary
if 'model_accuracy' in eval_m:
    for method, acc in eval_m['model_accuracy'].items():
        trained_results[method] = {'acc': acc}

if trained_results:
    methods = list(trained_results.keys())
    accs    = [trained_results[m].get('acc', 0) for m in methods]
    ax.bar(methods, accs, color=colors[:len(methods)], alpha=0.85,
           edgecolor='black', linewidth=0.6)
    for i, (meth, acc) in enumerate(zip(methods, accs)):
        ax.text(i, acc + 0.005, f'{acc:.3f}', ha='center', fontsize=11, fontweight='bold')
    ax.set_ylim(0.6, 1.05)
    ax.set_ylabel('Accuracy')
    ax.set_title('(b) Hasil Training Ini (Held-out Test Set)', fontsize=11)
    ax.grid(alpha=0.2, axis='y', linestyle=':')
else:
    ax.text(0.5, 0.5, 'Jalankan evaluasi (Step 6) terlebih dahulu',
            ha='center', va='center', transform=ax.transAxes, fontsize=11, color='gray')
    ax.set_title('(b) Hasil Training Ini — belum tersedia', fontsize=11)

plt.tight_layout()
plt.savefig('results/fig_final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan → results/fig_final_results.png')


---
## 💾 Download Hasil (Opsional)

In [ ]:
# ── Download hasil ke komputer lokal (opsional) ───────────────────────────────
# Uncomment cell ini jika mau download ZIP hasil training

# import shutil
# shutil.make_archive('/content/training_results', 'zip', 'results')
# from google.colab import files
# files.download('/content/training_results.zip')

# ── Atau download model saja ──────────────────────────────────────────────────
# shutil.make_archive('/content/trained_models', 'zip', 'models')
# from google.colab import files
# files.download('/content/trained_models.zip')

print('Semua file sudah tersimpan di Google Drive:')
print('  /MyDrive/duck_egg_fertility/models/')
print('  /MyDrive/duck_egg_fertility/results/')
print('  /MyDrive/duck_egg_fertility/data/features/')
print()
print('Uncomment kode di atas jika mau download ke lokal.')


---
## Ringkasan Pipeline Training

| Step | Script | Output | Estimasi Waktu |
|------|--------|--------|----------------|
| 1 | `01_preprocess_data.py` | `data/preprocessed/` | ~2 menit |
| 2 | `03b_generate_masks.py` | `data/segmentation/` | ~3 menit |
| 3 | `03_train_unet.py` | `models/unet/unet_best.pth` | ~20 menit (T4 GPU) |
| 4 | `04_extract_features.py` | `data/features/*.npy` | ~5 menit |
| 5 | `05_train_awc.py` | `models/awc/awc_model.pkl` | ~1 menit |
| 6 | `06_evaluate_models.py` | `results/evaluation/` | ~2 menit |

**Total estimasi: ~35 menit di T4 GPU**

### Catatan penting:
- Semua checkpoint otomatis disimpan ke Google Drive
- Jika sesi Colab putus, cukup mount Drive lagi dan lanjutkan dari step terakhir
- Model terbaik U-Net dipilih berdasarkan `val_loss` (early stopping patience=15)
